# PennyLane + `numpy_annotated`

PennyLane circuits return NumPy arrays (state vectors, probabilities, expectation values).
This notebook shows how to **validate** those arrays with Pydantic using our `NDArray[...]` annotations,
then **serialize** them to JSON for APIs or storage.

## Setup (once)

From the `xanady_presentation` folder in a terminal:

```bash
uv sync --extra notebook
```

## Pick the right Jupyter kernel in Cursor

Click **Select Kernel** (top-right of the notebook) and choose either:

- **`Python (xanady-presentation)`** — registered project kernel, or
- **`.venv (Python 3.x)`** — the interpreter at `xanady_presentation/.venv/Scripts/python.exe`

If you use a global/system Python kernel, `import numpy_annotated` will fail because the
library is installed in the project virtualenv, not globally.

The next code cell also adds `src/` to `sys.path` as a fallback when the kernel cwd is the repo.

In [1]:
# Fallback: add src/ if the kernel is not the project .venv
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
src = root / "src"
if src.is_dir() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import json

import numpy as np
import pennylane as qml
from pydantic import BaseModel, ValidationError

from numpy_annotated import NDArray, NDArrayConfig
print("numpy_annotated OK — project root:", root)

numpy_annotated OK — project root: c:\Users\andre\Downloads\Jobs\Xanadu Presentation\xanady_presentation


## 1. Build a small circuit

Two-qubit Bell state: the state vector lives in $\mathbb{C}^4$.

In [2]:
dev = qml.device("default.qubit", wires=2)


@qml.qnode(dev)
def bell_state():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.state()


state = bell_state()
print(f"dtype: {state.dtype}, shape: {state.shape}")
print(state)

dtype: complex128, shape: (4,)
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


## 2. Validate with Pydantic

The field type constrains **shape** `(4,)` and **dtype** `complex128` — mismatches raise `ValidationError`.

In [3]:
class BellStateVector(BaseModel):
    """2-qubit state vector in the computational basis."""

    amplitudes: NDArray[(4,), np.complex128]


model = BellStateVector(amplitudes=state)
print("Validated:", model.amplitudes)
print("Norm:", np.linalg.norm(model.amplitudes))

Validated: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Norm: 0.9999999999999999


In [4]:
# Wrong dtype is rejected
try:
    BellStateVector(amplitudes=state.real.astype(np.float64))
except ValidationError as exc:
    print("Rejected float64 state:", exc.errors()[0]["msg"])

## 3. JSON round-trip (complex matrix)

Gate and interferometer unitaries are **2×2 (or N×N) complex matrices**. List-format JSON encodes each entry as `[re, im]` so imaginary parts survive round-trip (unlike generic pydantic–numpy helpers).

In [5]:
# Pauli-Y: purely imaginary off-diagonal entries (good exercise for [re, im] JSON)
U = qml.matrix(qml.PauliY(wires=0))
print(f"U shape: {U.shape}, dtype: {U.dtype}")
print("U:\n", U)


class UnitaryPayload(BaseModel):
    matrix: NDArray[(2, 2), np.complex128]


payload_model = UnitaryPayload(matrix=U)
json_str = payload_model.model_dump_json()
parsed = json.loads(json_str)

print("JSON:", json.dumps(json_str))
print("shape:", parsed["matrix"]["shape"])
print("U[0,1] as [re, im]:", parsed["matrix"]["data"][0][1])  # -1j -> [0, -1]

restored = UnitaryPayload.model_validate_json(json_str)
print("Round-trip equal:", np.allclose(restored.matrix, U))
print("Imaginary part preserved:", np.allclose(restored.matrix.imag, U.imag))

# Re-serialize after deserialize (e.g. API receives JSON, then sends it onward)
json_str_2 = restored.model_dump_json()
print("Re-serialize identical:", json_str_2 == json_str)
restored_2 = UnitaryPayload.model_validate_json(json_str_2)
print("Second round-trip equal:", np.allclose(restored_2.matrix, U))

U shape: (2, 2), dtype: complex128
U:
 [[ 0.+0.j -0.-1.j]
 [ 0.+1.j  0.+0.j]]
JSON: "{\"matrix\":{\"dtype\":\"<c16\",\"shape\":[2,2],\"data\":[[[0.0,0.0],[-0.0,-1.0]],[[0.0,1.0],[0.0,0.0]]]}}"
shape: [2, 2]
U[0,1] as [re, im]: [-0.0, -1.0]
Round-trip equal: True
Imaginary part preserved: True
Re-serialize identical: True
Second round-trip equal: True


### Base64 encoding

For large operators (e.g. N×N interferometers), nested `[re, im]` JSON grows quickly. Pass `NDArrayConfig(to_base64=True)` to serialize the raw buffer as a base64 string — compact and **bit-exact** for `complex128`.

In [7]:
class UnitaryPayloadB64(BaseModel):
    matrix: NDArray[
        ((2, 2), np.complex128, NDArrayConfig(to_base64=True))
    ]


payload_b64 = UnitaryPayloadB64(matrix=U)
json_b64 = payload_b64.model_dump_json()
parsed_b64 = json.loads(json_b64)

print("dtype:", parsed_b64["matrix"]["dtype"])
print("shape:", parsed_b64["matrix"]["shape"])
print("data is base64:", isinstance(parsed_b64["matrix"]["data"], str))
print("data preview:", parsed_b64["matrix"]["data"][:48], "...")

restored_b64 = UnitaryPayloadB64.model_validate_json(json_b64)
print("Round-trip bit-exact:", np.array_equal(restored_b64.matrix, U))
print("Re-serialize identical:", restored_b64.model_dump_json() == json_b64)

dtype: <c16
shape: [2, 2]
data is base64: True
data preview: AAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAAAAAAAA8L8AAAAA ...
Round-trip bit-exact: True
Re-serialize identical: True


## 4. Measurement probabilities (real vector)

Many pipelines use **probability vectors** (`float64`) instead of full amplitudes.

In [8]:
@qml.qnode(dev)
def bell_probs():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])


probs = bell_probs()
print(f"probs: {probs}")


class MeasurementRecord(BaseModel):
    probabilities: NDArray[(4,), np.float64]


record = MeasurementRecord(probabilities=probs)
print("Sum of probabilities:", record.probabilities.sum())

probs: [0.5 0.  0.  0.5]
Sum of probabilities: 0.9999999999999998


## 5. Gate matrix as a validated operator

PennyLane's `qml.matrix` builds the dense unitary for a gate — the same pattern applies to photonic / linear-optics operators stored as `NDArray[("N", "N"), np.complex128]`.

In [9]:
theta = 0.3
U = qml.matrix(qml.RY(theta, wires=0))
print(f"U shape: {U.shape}, dtype: {U.dtype}")


class SingleQubitUnitary(BaseModel):
    matrix: NDArray[(2, 2), np.complex128]


op = SingleQubitUnitary(matrix=U)
print("Unitary:\n", op.matrix)

U shape: (2, 2), dtype: complex128
Unitary:
 [[ 0.98877108+0.j -0.14943813-0.j]
 [ 0.14943813+0.j  0.98877108+0.j]]


## Summary

| PennyLane output | `NDArray` typing |
|------------------|------------------|
| `qml.state()` on *n* qubits | `NDArray[(2**n,), np.complex128]` or `(None,)` |
| `qml.probs()` | `NDArray[(None,), np.float64]` |
| `qml.matrix(...)` | `NDArray[("N", "N"), np.complex128]` for square ops |

The pattern is the same for photonic simulators (e.g. Strawberry Fields): circuit → NumPy array → Pydantic model with dtype/shape checks → JSON.